# 04 - Metadata preflight

- **Meeting item:** Paired two-view preflight
- **Commit:** C06 - `feat: add metadata preflight schema and reporter`
- **Commit achievement:** makes dataset feasibility inspectable before any training decision.
- **Data mode:** `synthetic`. Every table and file below is generated in this notebook. Nothing is downloaded.
- **Evidence labels used:** verified, unknown
- **Human gate:** can a reviewer answer the donor, label, pairing, missingness, and modality-shape questions from the report?
- **Stop condition:** missing required metadata is guessed instead of reported as `unknown`.

Logic lives in `src/p22/data/schema.py`, `src/p22/data/preflight.py`, and `src/p22/cli/preflight.py`.
When an approved dataset exists, the same command runs against it unchanged.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import p22
from p22.data.preflight import inspect_h5ad, inspect_metadata, write_preflight_report
from p22.data.schema import ACCESS_FACTS, DEFAULT_SCHEMA, UNKNOWN
from p22.testing import make_synthetic_multimodal

dataset = make_synthetic_multimodal(n_donors=6, cells_per_donor=8, seed=11)
metadata = dataset.metadata.copy()

print(f"required columns: {DEFAULT_SCHEMA.required}")
print(f"optional columns: {DEFAULT_SCHEMA.optional}")
print(f"facts that cannot be inferred from a matrix: {ACCESS_FACTS}")
metadata.head()

## 1. Honest default: unknown, not assumed (evidence label: verified)

With no pairing information and no recorded access facts, the report is `INCONCLUSIVE`. It
does not guess that the views are paired and it does not assume the data may be used.

In [ ]:
bare = inspect_metadata(metadata)
print("\n".join(bare.summary_lines()))
print("")
print(f"access facts as reported: {bare.sections['access_facts']}")

## 2. The five meeting questions (evidence label: verified)

Donors, labels, pairing, missingness, and modality shape, each answered from the report.

In [ ]:
recorded_facts = {
    "licence": "synthetic fixture, no licence applies",
    "access_level": "open, generated locally",
    "consent_terms": "not applicable to generated arrays",
    "redistribution_allowed": "yes, generated locally",
    "controlled_access_approval": "not required for generated arrays",
}

report = inspect_metadata(
    metadata,
    view_shapes={"view_a": dataset.view_a.shape, "view_b": dataset.view_b.shape},
    view_cell_ids={"view_a": dataset.cell_ids, "view_b": dataset.cell_ids},
    access_facts=recorded_facts,
)

print(f"status: {report.status}")
print(f"donor question: {report.sections['donors']}")
print(f"label question: {report.sections['labels']}")
print(f"pairing question: {report.sections['pairing']}")
print(f"modality question: {report.sections['modalities']}")
print(f"duplicate identifiers: {report.sections['duplicates']}")

In [ ]:
print("missingness per column:")
print(pd.DataFrame(report.sections["missingness"]).T.to_string())
print("")
print("per-donor class counts:")
print(pd.DataFrame(report.sections["per_donor_class_counts"]["counts"]).T.fillna(0).astype(int).to_string())

## 3. What blocks a dataset (evidence label: verified)

Each row below is a separate inspection of a deliberately damaged copy of the same table.

In [ ]:
paired = {
    "view_shapes": {"view_a": dataset.view_a.shape, "view_b": dataset.view_b.shape},
    "view_cell_ids": {"view_a": dataset.cell_ids, "view_b": dataset.cell_ids},
    "access_facts": recorded_facts,
}

no_donor = metadata.drop(columns=["donor_id"])
duplicated = metadata.copy()
duplicated.loc[1, "cell_id"] = duplicated.loc[0, "cell_id"]
missing_label = metadata.copy()
missing_label.loc[2, "label"] = np.nan

cases = {
    "donor column absent": inspect_metadata(no_donor, access_facts=recorded_facts),
    "duplicate cell identifiers": inspect_metadata(duplicated, **paired),
    "missing label value": inspect_metadata(missing_label, **paired),
    "view row count disagrees": inspect_metadata(
        metadata,
        view_shapes={"view_a": (47, 16), "view_b": dataset.view_b.shape},
        view_cell_ids={"view_a": dataset.cell_ids, "view_b": dataset.cell_ids},
        access_facts=recorded_facts,
    ),
    "views share no cells": inspect_metadata(
        metadata,
        view_shapes={"view_a": dataset.view_a.shape, "view_b": dataset.view_b.shape},
        view_cell_ids={"view_a": dataset.cell_ids[:24], "view_b": dataset.cell_ids[24:]},
        access_facts=recorded_facts,
    ),
    "views overlap only partly": inspect_metadata(
        metadata,
        view_shapes={"view_a": dataset.view_a.shape, "view_b": dataset.view_b.shape},
        view_cell_ids={"view_a": dataset.cell_ids, "view_b": dataset.cell_ids[:40]},
        access_facts=recorded_facts,
    ),
}

rows = []
for description, case in cases.items():
    rows.append(
        {
            "case": description,
            "status": case.status,
            "first_finding": (case.blocking_problems or case.open_questions)[0],
        }
    )
pd.DataFrame(rows)

## 4. File-level inspection (evidence label: verified)

A tiny `.h5ad` file is written to a temporary directory in this notebook and inspected from
disk. Remote paths are refused, so the same call cannot become a download.

In [ ]:
import anndata

with tempfile.TemporaryDirectory() as directory:
    obs = pd.DataFrame(
        {
            "donor": metadata["donor_id"].to_numpy(),
            "label": metadata["label"].to_numpy(),
        },
        index=pd.Index(dataset.cell_ids.astype(str), name="cell_id"),
    )
    path = Path(directory) / "tiny_two_view_fixture.h5ad"
    anndata.AnnData(X=dataset.view_a.astype("float32"), obs=obs).write_h5ad(path)
    print(f"wrote {path.name}, {path.stat().st_size / 1024:.1f} KiB")
    file_report = inspect_h5ad(path, donor_column="donor", access_facts=recorded_facts)

print("\n".join(file_report.summary_lines()))
print("")
for remote in ("https://example.org/matrix.h5ad", "s3://bucket/matrix.h5ad"):
    try:
        inspect_h5ad(remote)
    except ValueError as error:
        print(f"refused {remote}: {error}")

## 5. Written report (evidence label: verified)

Reports are written under the ignored `reports/generated/` tree, never into the source tree.

In [ ]:
out_path = p22.REPO_ROOT / "reports" / "generated" / "preflight" / "synthetic_fixture.json"
written = write_preflight_report(report, out_path)
payload = json.loads(written.read_text())
print(f"written: {written.relative_to(p22.REPO_ROOT)}")
print(f"top-level keys: {sorted(payload)}")
print(f"status in file: {payload['status']}")
print(f"data mode in file: {payload['data_mode']}")
print(f"network access in file: {payload['network_access']}")

In [ ]:
assert report.status == "PASS"
assert report.sections["pairing"]["status"] == "paired"
assert report.sections["donors"]["n_donors"] == 6
assert report.sections["labels"]["n_classes"] == 3
assert report.sections["rows"]["n_rows"] == 48
assert bare.status == "INCONCLUSIVE"
assert bare.sections["pairing"]["status"] == UNKNOWN
assert all(value == UNKNOWN for value in bare.sections["access_facts"].values())
assert cases["donor column absent"].status == "BLOCKED"
assert cases["duplicate cell identifiers"].status == "BLOCKED"
assert cases["missing label value"].status == "BLOCKED"
assert cases["views share no cells"].status == "BLOCKED"
assert file_report.sections["pairing"]["status"] == UNKNOWN
print("all C06 gate checks passed")

## 6. Unknown (evidence label: unknown)

- Which dataset will be approved, and therefore whether donors, labels, and two paired views exist in it.
- Licence, access level, consent terms, redistribution, and controlled-access approval for any such dataset. These stay `unknown` until a human records them; the reporter will not fill them in.
- Whether a partially paired dataset would leave enough cells to be usable. That decision belongs to a human, not to this report.

## Gate

**PASS**

- **Observed:** the report answers all five questions for a 48-cell, 6-donor, 3-class fixture with two views of 16 and 12 features; with pairing and access facts absent the status is `INCONCLUSIVE` with explicit `unknown` values rather than guesses; an absent donor column, duplicate cell identifiers, a missing label value, a view row-count disagreement, and disjoint views each produce `BLOCKED`; a partially paired pair produces `INCONCLUSIVE`; a locally written tiny `.h5ad` file is inspected from disk and remote paths are refused; the JSON report is written under the ignored `reports/generated/` tree.
- **Remains unknown:** which dataset will be approved and all of its licence and access facts. The reporter records them as `unknown` until a human fills them in.
- **Next decision owner:** researcher continues with C07 (model components). Running this preflight on a candidate dataset needs the named approver first.